# Task 4 -- Open-Set Recognition on CIFAR-10 / CIFAR-100

**Before running:** Settings (right sidebar) -> Accelerator: **GPU T4 x2**, Internet: **On**.

**Inputs you need attached** (Add Input, right sidebar):
1. `task4_pa1.zip` -- this task's code, uploaded as a Kaggle dataset (same way as `task1_pa1.zip` / `task2_pa1.zip` / `task3_pa1.zip`).

CIFAR-10 and CIFAR-100 are NOT attached as Kaggle datasets -- they download
automatically via `torchvision.datasets` the first time this notebook runs
(needs Internet: On), so there is no PACS-style dataset hunting this time.

In [ ]:
!nvidia-smi
!curl -sI https://www.google.com | head -1

In [ ]:
# --- Locate this task's code (os.walk-based -- glob's recursive "**" was
#     observed to miss files under Kaggle's nested datasets/<owner>/<slug>/
#     mount layout in Task 3, so this notebook uses os.walk from the start) ---
import os, zipfile, shutil

def find_file(root, filename):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        if filename in filenames:
            hits.append(os.path.join(dirpath, filename))
    return hits

hits = find_file("/kaggle/input", "evaluate_osr.py")
if hits:
    src_root_dir = os.path.dirname(os.path.dirname(hits[0]))
    print("Found already-extracted code at:", src_root_dir)
    os.makedirs("/kaggle/working/pa4", exist_ok=True)
    shutil.copytree(src_root_dir, "/kaggle/working/pa4", dirs_exist_ok=True)
else:
    zip_candidates = []
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", followlinks=True):
        for fn in filenames:
            if fn.lower().endswith(".zip") and ("task4" in fn.lower() or "pa4" in fn.lower()):
                zip_candidates.append(os.path.join(dirpath, fn))
    assert zip_candidates, "Could not find task4/ (extracted) or task4_pa1.zip under /kaggle/input."
    code_zip = zip_candidates[0]
    print("Using code zip:", code_zip)
    os.makedirs("/kaggle/working/pa4", exist_ok=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/kaggle/working/pa4")

hits2 = find_file("/kaggle/working/pa4", "evaluate_osr.py")
assert hits2, "task4/evaluate_osr.py not found after copy/extract -- check input contents."
task4_dir = os.path.dirname(hits2[0])
print("task4/ ready at:", task4_dir)

%cd {task4_dir}
print("Working directory:", os.getcwd())


In [ ]:
# --- Point config.yaml at a writable, persistent-for-this-session data dir
#     (CIFAR-10/100 download here automatically on first use) ---
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["dataset"]["root"] = "/kaggle/working/data"
with open("configs/config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("dataset.root set to:", cfg["dataset"]["root"])

In [ ]:
# --- Smoke test (seconds, no real data needed) -- run this before anything else ---
!python tests/test_pipeline.py

If the smoke test above shows any `[FAIL]` lines, stop here before training -- everything below burns real GPU time.

In [ ]:
# Downloads CIFAR-10 (~170MB) and CIFAR-100 (~170MB) on first run.
!python train.py --method vanilla

In [ ]:
!python train.py --method gcsc

In [ ]:
!python train.py --method proser --vanilla_checkpoint checkpoints/vanilla.pt

### Cache features/logits for every method (CIFAR-10 train/val/test + the fixed CIFAR-100 near/far groups)
Each checkpoint is already frozen at this point -- this step never trains or selects anything.

In [ ]:
!python extract_outputs.py --method vanilla
!python extract_outputs.py --method gcsc
!python extract_outputs.py --method proser

In [ ]:
# THE SCRIPT THAT ACTUALLY COMPUTES SCORES, AUROC, AND THRESHOLDS
!python evaluate_osr.py

In [ ]:
# --- Zip everything you need to download ---
import shutil
shutil.make_archive("/kaggle/working/task4_results", "zip", ".", "results")
shutil.make_archive("/kaggle/working/task4_checkpoints", "zip", ".", "checkpoints")
print("Download task4_results.zip and task4_checkpoints.zip from the Output pane on the right.")